# Language model and sequence generation

In this lab, we will study the different ways of decoding text from a trained model. This step is needed for many natural language processing tasks: translation, question answering, generation…

We will take the example of text generation with GPT-2, a *decoder-only* transformer: today's large language models (GPT-4, Llama, Mistral…) rely on the same principle and are decoded with the same methods.

We will start from the simplest method and work our way up to the methods used in practice, writing the choice of the next token ourselves. This will help you understand the role of the softmax at the output of language models. To finish, we will find all these settings in the `generate` method of the `transformers` library.

An [excellent article](https://huggingface.co/blog/how-to-generate) by Hugging Face can serve as a complementary resource to this notebook.

This lab doesn't need a GPU: GPT-2 small runs on CPU, where each test on the lab's 7 prompts takes less than a minute. It uses the GPU if there is one.

**Warning**: these models were trained on web text, without filtering. The generated texts, especially by sampling, can be incoherent, false or offensive.

## Installation

The [`transformers`](https://github.com/huggingface/transformers) library by [Hugging Face](https://huggingface.co/) provides a large number of models of the transformer family. It is preinstalled on Colab; the following cell installs it elsewhere (Codespace, personal computer).

In [ ]:
!pip install -q transformers

## Choice of language

We will see how to generate text in English but also in French, by loading models pretrained on different corpora. The following cell changes the language used for the rest of the lab.

In [ ]:
lang = "en"
# lang = "fr"

## Loading a trained model

Besides providing the source code of many transformer architectures, the `transformers` library fetches [trained models](https://huggingface.co/models) from the Hugging Face Hub. The `from_pretrained` method of `AutoModelForCausalLM` fetches the architecture and the weights of a model that predicts the next token (*causal language model*), that of `AutoTokenizer` its tokenizer.

- In English, `"gpt2"` is OpenAI's GPT-2 small (2019).
- In French, [BelGPT-2](https://huggingface.co/antoinelouis/belgpt2) is a GPT-2 small trained on a French corpus.

PyTorch models are `nn.Module`s: we move them to the GPU with `.to(device)`, and switch them to evaluation mode with `.eval()` (dropout is then disabled).

The texts a GPT-2 sees during its training are separated by a special beginning and end of text token: `<|endoftext|>` for GPT-2, `</s>` for BelGPT-2. The cell keeps its index in `start_token_id`.

In [ ]:
import functools
import statistics

import torch
import tqdm.auto
import transformers

# Same random draws at each run of the notebook
torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Computing on: {device}")

pretraining_name = "antoinelouis/belgpt2" if lang == "fr" else "gpt2"

model = transformers.AutoModelForCausalLM.from_pretrained(pretraining_name)
model = model.to(device).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(pretraining_name)

# Beginning and end of text token
start_token_id = tokenizer.convert_tokens_to_ids(
  "</s>" if lang == "fr" else "<|endoftext|>"
)

## Inspecting the model

After loading a pretrained model, it is always a good idea to check its architecture and characteristics. To do so, we can display its configuration and the model itself (an `nn.Module` is displayed with all its layers).

- *How many transformer layers does the model have?*
- *What are its regularization mechanisms?*
- *Its training stabilization mechanisms?*
- *How many parameters does it have?*
- *The configuration gives the index of the beginning of text token in `bos_token_id`. Is it `start_token_id`? Display the token that each one stands for with `tokenizer.convert_ids_to_tokens`, in English then in French.*

In [ ]:
print(model.config)
print(model)

*Your answer here.*
-
-
-
-
-

### Solution

In [ ]:
n_parameters = sum(p.numel() for p in model.parameters())
print(f"{n_parameters / 1e6:.0f} million parameters")
print(f"Layers: {model.config.n_layer}")
for name, index in [
  ("bos_token_id", model.config.bos_token_id),
  ("start_token_id", start_token_id),
]:
  print(f"{name}: {index} = {tokenizer.convert_ids_to_tokens(index)!r}")

- It has 12 transformer layers (`n_layer`, or the 12 `GPT2Block`s displayed).
- Its main regularization mechanism is dropout (`attn_pdrop`, `resid_pdrop`, `embd_pdrop`).
- Its training stabilization mechanisms are layer normalization (`LayerNorm`, `ln_1`, `ln_2`) and residual connections.
- It has about 124 million parameters. The output layer `lm_head` adds none: it reuses the input's token embedding matrix (`wte`), and `parameters()` counts it only once.
- For GPT-2, `bos_token_id` is 50256, `<|endoftext|>`: it is indeed `start_token_id`. For BelGPT-2, `bos_token_id` is 50256 too, but this token is `ĠArcade` in its vocabulary: its configuration kept GPT-2's value, whereas its beginning and end of text token is `</s>`, of index 2. A pretrained model isn't always consistent with its tokenizer: check its special tokens before using them.

## The generation mechanism

To generate text from a trained model, we proceed iteratively: we generate the tokens one by one, starting from an empty text or from a sentence we want to complete.

To generate a token, we feed the network the tokens generated so far (as indices in the vocabulary). The network then outputs one score per vocabulary token. We pick one of these tokens, append it to what has already been generated, and start again to generate the next token.

More formally, we try to maximize $P(\text{Text}| \text{Initial})$ with the following decomposition:

$$P(\text{Text}| \text{Initial}) = \prod_{i=1}^{\text{Text length}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

The whole point of decoding is to find the text that maximizes this product: it is the model's best proposal. This value can't be computed exactly, so we use heuristics to approximate it.

The transformer's **causal mask** is what makes this decomposition possible: during training, each position only sees the previous positions.

## Tokenization

The tokenizer loaded with the model turns a sentence into vocabulary indices (*subwords*) the model understands. In the display of the tokens, `Ġ` marks a token that starts with a space.

In the rest of the lab, we use the following `encode` and `decode` functions. `encode` adds the beginning of text token, and returns a tensor of shape `(1, number of tokens)` on the right device.

BelGPT-2 was trained on texts where punctuation is separated from words by spaces ("j' ai", "fin ."): its generations keep a trace of it.

In [ ]:
def encode(sentence: str) -> torch.Tensor:
  """Token indices, after the beginning token: shape (1, number of tokens)."""
  tokens = tokenizer.encode(sentence, add_special_tokens=False, return_tensors="pt")
  start = torch.tensor([[start_token_id]])
  return torch.cat([start, tokens], dim=1).to(device)


def decode(tokens: torch.Tensor) -> str:
  """Text of the tokens, without the beginning token."""
  return tokenizer.decode(tokens.squeeze(0)[1:])


example = encode(
  "Je pense, donc je suis" if lang == "fr" else "I think, therefore I am"
)
print(example)
print(tokenizer.convert_ids_to_tokens(example[0]))

## Studying the outputs of the pretrained model

The first step to decode sentences from a trained model is to run a forward pass of the model to get its predictions for the next token. Let's study its behavior.

As with any PyTorch module, the forward pass is computed with `model(token_ids)`. The result is an object whose main attributes are:

- `logits`: the model's scores, before softmax;
- `past_key_values`: a cache of the attention keys and values already computed.

*Apply the model to various sequences, for example:*

- *"" (the empty text)*
- *"Je pense, donc je suis" or "I think, therefore I am" depending on the model's language*

*What is the shape of the `logits`? What does each dimension stand for?*

*Disable gradient computation: we are not training the model.*

**Useful documentation**

- [`GPT2LMHeadModel`](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel)
- [`torch.no_grad`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

In [ ]:
# Your code here

### Additional hints

The calls to use:

- `encode(sentence)` gives the `token_ids`, of shape `(1, number of tokens)`
- `with torch.no_grad():`: the computations of the block keep nothing to compute gradients with
- `output = model(token_ids)`, then `output.logits.shape` gives the shape of the logits
- Expected result for the empty text: `(1, 1, 50257)`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def describe_shapes(sentence: str) -> None:
  with torch.no_grad():
    output = model(...)
  print(f"Shape of the logits for '{sentence}': {tuple(...)}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")
```

### Solution

In [ ]:
def describe_shapes(sentence: str) -> None:
  with torch.no_grad():
    output = model(encode(sentence))
  print(f"Shape of the logits for '{sentence}': {tuple(output.logits.shape)}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")

The logits have the shape `(batch, number of tokens, vocabulary size)`: for each position of the input, the model gives a score to each of the 50,257 tokens of the vocabulary for the **next** position. The empty text has one position, that of the beginning token `encode` adds. To generate, only the last position matters.

## A `forward` function suited to decoding

We will use the following `forward` function, which returns a result suited to decoding: only the last position matters (the prediction of the next token).

It uses the `past_key_values` cache: it avoids recomputing, at each new token, the attention keys and values of all the previous tokens. When the cache is given, only **the new tokens** must be passed to the model.

In [ ]:
@torch.no_grad()
def forward(tokens: torch.Tensor, past=None) -> tuple[torch.Tensor, object]:
  """Logits of the next token, of shape (1, vocabulary size), and the cache."""
  output = model(tokens, past_key_values=past, use_cache=True)
  return output.logits[:, -1, :], output.past_key_values

So use either:

- `forward(all_tokens)`
- `forward(last_token, past)`

where `all_tokens` would be for example a sequence of 7 tokens of shape `(1, 7)` while `last_token` would be of shape `(1, 1)`. The logits returned have the shape `(1, vocabulary size)`: the batch dimension, of size 1, is kept.

*Why is it often useful to keep dimensions of size `1` in neural networks?*

*Your answer here.*

### Solution

These dimensions make it possible to compute results for several examples in parallel (batching): the same code works for a batch of 1 as well as of 64.

## Decoding loop

A decoding loop always follows the same pattern:

1. Forward pass on what has been produced so far (inputs and previous outputs; thanks to the cache, only the last token after the first step)
2. Choice of the index of the next token
3. Repetition of 1. and 2. until a stopping criterion is met (in this lab, we will only use the number of tokens produced)

Step 2 is where everything happens. The following code implements all the rest, so that we can focus on step 2: each decoding method will be a function that receives the logits of the last token (shape `(1, vocabulary size)`) and returns the index of the chosen token (a tensor of shape `()`). Decorated with `@decoding_loop`, it becomes a function `(prompt, number of tokens to generate, parameters of the method)` that returns the generated text.

In [ ]:
def decoding_loop(step_function):

  @functools.wraps(step_function)
  def wrapper(prompt: str, length: int, *step_args, **step_kwargs) -> str:
    decoded = [encode(prompt)]
    past = None
    for _ in tqdm.auto.trange(length, desc="Tokens", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded.append(index.reshape(1, 1))
    return decode(torch.cat(decoded, dim=1))

  return wrapper

## Greedy decoding

Greedy decoding is the simplest method to decode text from a trained model: at each step, we take the token of highest probability.

In the equation:

$$P(\text{Text}|\text{Initial}) = \prod_{i=1}^{\text{Text length}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

greedy decoding chooses $\text{Token}_i$ to maximize $P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$. Yet one often has to choose a less likely token to then reach very likely tokens.

The following example comes from Hugging Face's article on text generation:

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

After "The", we choose "nice" because it is the most likely word, whereas the complete text "The nice woman" has a probability (0.20) lower than the one we would have got by choosing "dog" ("The dog has", 0.36).

*Write the function `greedy(logits)` that takes the model's logits as input and returns the index of the chosen token, a tensor of shape `()`. It will use `torch.argmax`. Generate 10 tokens after "I went to the" ("Je suis allé au" in French).*

**Useful documentation**

- [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

In [ ]:
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `torch.argmax(logits, dim=-1)`: the index of the largest logit of each row, of shape `(1,)`; `[0]` extracts the tensor of shape `()` from it
- No need for the softmax: it doesn't change the order of the scores
- `greedy(prompt, 10)`: the decorated function takes the prompt and the number of tokens to generate

A skeleton to copy into the exercise cell, replacing each `...`:

```python
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor:
  return ...


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))
```

### Solution

In [ ]:
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor:
  return torch.argmax(logits, dim=-1)[0]


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))

## Test on varied prompts

The following `test_decoding` function applies a decoding method, with its parameters, to each prompt of the `inputs` list. It displays the generations and returns them in a list; `show` displays them.

*Run the following cells, which test greedy decoding with 50 tokens per prompt. What do you notice?*

In [ ]:
if lang == "fr":
  inputs = [
    "Je suis allé au",
    "Le train pour Lyon",
    "Comment vas-tu",
    "La recette de la tarte aux pommes commence par",
    "Ça va merci, tu devrais",
    "Le chat de la voisine",
    "En 2050, les voitures",
  ]
else:
  inputs = [
    "I went to the",
    "The train to London",
    "How do you",
    "The apple pie recipe starts with",
    "I'm fine thank you, you should",
    "The neighbour's cat",
    "In 2050, cars",
  ]

In [ ]:
def show(generations: list[str]) -> None:
  """Display each prompt of `inputs` with its generation."""
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt: {prompt}")
    print(f"Generation: {generation}")
  print("—" * 80)


def test_decoding(function, *args, **kwargs) -> list[str]:
  """Apply the decoding method `function` to each prompt of `inputs`."""
  generations = [
    function(prompt, *args, **kwargs)
    for prompt in tqdm.auto.tqdm(inputs, desc="Prompts", leave=False)
  ]
  show(generations)
  return generations

In [ ]:
greedy_generations = test_decoding(greedy, 50)

## Measuring repetition

Greedy generations quickly "loop": they repeat the same groups of words. To compare decoding methods on this point, let's measure it with the **share of distinct 3-grams** of a text: we split the text into words, form all the triples of consecutive words (the 3-grams), and divide the number of different triples by the total number of triples. It is 1 for a text without repetition, and goes down when passages repeat.

*Write the function `distinct_3grams(text)` that returns the share of distinct 3-grams of `text`. Then display its mean over the greedy generations, `greedy_generations`, with `statistics.mean`. The next sections reuse it for each method.*

In [ ]:
def distinct_3grams(text: str) -> float: ...


# Your code here

### Additional hints

The calls to use:

- `text.split()`: the list of words, separated by spaces
- `zip(words, words[1:], words[2:])`: the triples of consecutive words
- `set(...)`: the different triples
- `statistics.mean(map(distinct_3grams, greedy_generations))`: the mean over the generations
- Example to check: `"a b c a b c"` has 4 triples, 3 of them different: 0.75

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def distinct_3grams(text: str) -> float:
  words = text.split()
  trigrams = list(...)
  return ... / ...


print(distinct_3grams("a b c a b c"))
print(f"Greedy: {statistics.mean(...):.2f}")
```

### Solution

In [ ]:
def distinct_3grams(text: str) -> float:
  words = text.split()
  trigrams = list(zip(words, words[1:], words[2:]))
  return len(set(trigrams)) / len(trigrams)


print(distinct_3grams("a b c a b c"))
print(f"Greedy: {statistics.mean(map(distinct_3grams, greedy_generations)):.2f}")

Several greedy generations repeat the same sentence, or the same group of words, until the end: this is a very classic weakness of greedy decoding. The share of distinct 3-grams puts a number on it: it is well below 1, where a text written by a human is close to it.

## Sampling

The results of greedy decoding loop quickly and are often very generic. The following methods try to fix these flaws.

The first one is to **sample** from the model's probabilities, rather than always choosing the most likely token: a token of probability 0.3 is drawn 3 times out of 10.

*Take the shape of the `greedy` function, but instead of choosing the maximum element, compute a probability distribution with `torch.softmax`, then draw the next token according to this distribution with `torch.multinomial`, in a function `sampling(logits)`. Test it with `test_decoding` (50 tokens per prompt), and compare its share of distinct 3-grams with that of greedy decoding.*

**Useful documentation**

- [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)
- [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)

In [ ]:
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `torch.softmax(logits, dim=-1)`: the probabilities, of shape `(1, vocabulary size)`
- `torch.multinomial(probabilities, num_samples=1)`: one draw per row, of shape `(1, 1)`; `[0, 0]` extracts the tensor of shape `()` from it
- `statistics.mean(map(distinct_3grams, generations))`: the share of distinct 3-grams

A skeleton to copy into the exercise cell, replacing each `...`:

```python
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor:
  probabilities = ...
  return torch.multinomial(...)[0, 0]


generations = test_decoding(sampling, 50)
print(f"Distinct 3-grams: {...:.2f}")
```

### Solution

In [ ]:
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor:
  probabilities = torch.softmax(logits, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


generations = test_decoding(sampling, 50)
print(f"Distinct 3-grams: {statistics.mean(map(distinct_3grams, generations)):.2f}")

The generations no longer loop: the share of distinct 3-grams is close to 1. On the other hand, they often lose the thread, because an unlikely token is drawn from time to time and the rest of the text has to cope with it.

## Temperature

Before the softmax, the logits can be divided by a **temperature** $T$:

- $T < 1$ sharpens the differences: the model chooses the most likely tokens more often (more coherent, less varied);
- $T > 1$ flattens them: rare tokens are drawn more often (more varied, less coherent);
- $T \to 0$ amounts to greedy decoding.

*Add a `temperature` parameter to your sampling function, in a new function `temperature_sampling(logits, temperature)`. Compare the generations and their share of distinct 3-grams for $T = 0.5$ and $T = 1.5$.*

In [ ]:
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `logits / temperature`: the logits divided by the temperature, before the softmax
- `temperature_sampling(prompt, 50, temperature=0.5)`: the decorated function passes its keyword arguments on to yours; `test_decoding` too

A skeleton to copy into the exercise cell, replacing each `...`:

```python
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor:
  probabilities = torch.softmax(..., dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = test_decoding(temperature_sampling, 50, temperature=temperature)
  print(f"T = {temperature}: {...:.2f} distinct 3-grams")
```

### Solution

In [ ]:
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor:
  probabilities = torch.softmax(logits / temperature, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = test_decoding(temperature_sampling, 50, temperature=temperature)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"T = {temperature}: {repetition:.2f} distinct 3-grams")

At $T = 0.5$, the generations are more coherent, without looping yet over 50 tokens; the lower $T$, the closer they get to greedy decoding and its loops. At $T = 1.5$, they hardly make sense anymore: rare tokens are drawn at every step. The temperature sets this trade-off between coherence and variety.

## Top-k sampling

A variation of sampling is to draw only among the `k` most likely tokens, to avoid generating a very unlikely token.

*Start again from the `sampling` function to write the function `k_sampling(logits, k)` that implements this improvement with `torch.topk`. Test it with `test_decoding`, for `k = 20`.*

**Useful documentation**

- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)

In [ ]:
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `values, indices = torch.topk(logits, k=k, dim=-1)`: the `k` largest logits, in decreasing order, and their indices in the vocabulary, both of shape `(1, k)`
- `torch.softmax(values, dim=-1)`: the probabilities of the `k` tokens kept
- `torch.multinomial(probabilities, num_samples=1)[0, 0]`: the position drawn among the `k`
- `indices[0, position]`: the index in the vocabulary of the token drawn

A skeleton to copy into the exercise cell, replacing each `...`:

```python
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor:
  values, indices = torch.topk(...)
  probabilities = torch.softmax(..., dim=-1)
  sampled = torch.multinomial(probabilities, num_samples=1)[0, 0]
  return indices[...]


generations = test_decoding(k_sampling, 50, k=20)
print(f"Distinct 3-grams: {statistics.mean(map(distinct_3grams, generations)):.2f}")
```

### Solution

In [ ]:
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values, dim=-1)
  sampled = torch.multinomial(probabilities, num_samples=1)[0, 0]
  return indices[0, sampled]


generations = test_decoding(k_sampling, 50, k=20)
print(f"Distinct 3-grams: {statistics.mean(map(distinct_3grams, generations)):.2f}")

## Top-p sampling

Another improvement is to keep only the most likely tokens whose cumulative probability reaches `p` (the *nucleus* of the distribution), and not the next ones. The number of candidate tokens then adapts to the model's confidence: few candidates when one token dominates, many when the distribution is flat.

*Start again from the `k_sampling` function and add top-p sampling to it, in the function `k_p_sampling(logits, k, p)`. You can use `torch.cumsum`. Test it with `test_decoding`, for `k = 20` and `p = 0.85`.*

**Useful documentation**

- [`torch.cumsum`](https://docs.pytorch.org/docs/stable/generated/torch.cumsum.html)

In [ ]:
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `torch.softmax(values[0], dim=-1)`: the probabilities of the `k` tokens kept, in decreasing order, of shape `(k,)`
- `torch.cumsum(probabilities, dim=0) < p`: true for the tokens before the threshold; `.sum()` counts them, and we add 1 to keep the token that crosses `p`
- `int(...)` turns this number into a Python integer, to use it in a slice
- Example to check: for the probabilities `[0.5, 0.3, 0.15, 0.05]` and `p = 0.85`, the cumulative sums are `[0.5, 0.8, 0.95, 1.0]`: 2 are below `p`, we keep 3 tokens
- `torch.multinomial(probabilities[:n_kept], num_samples=1)[0]` draws among the first `n_kept`; `torch.multinomial` doesn't need probabilities that sum to 1

A skeleton to copy into the exercise cell, replacing each `...`:

```python
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = ...
  n_kept = int((...).sum()) + 1
  sampled = torch.multinomial(..., num_samples=1)[0]
  return indices[0, sampled]


generations = test_decoding(k_p_sampling, 50, k=20, p=0.85)
print(f"Distinct 3-grams: {statistics.mean(map(distinct_3grams, generations)):.2f}")
```

### Solution

In [ ]:
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values[0], dim=-1)
  # Number of tokens needed for the cumulative probability to reach p
  n_kept = int((torch.cumsum(probabilities, dim=0) < p).sum()) + 1
  sampled = torch.multinomial(probabilities[:n_kept], num_samples=1)[0]
  return indices[0, sampled]


generations = test_decoding(k_p_sampling, 50, k=20, p=0.85)
print(f"Distinct 3-grams: {statistics.mean(map(distinct_3grams, generations)):.2f}")

## Generating with `generate`

In practice, we don't write the decoding loop: the `generate` method of `transformers` models implements all these strategies, and others, such as beam search. It takes the indices of the prompt and an attention mask of the same shape (1 for a real token, 0 for a padding position: here, only 1s), and returns the indices of the prompt followed by the generated tokens. Its parameters set the choice of the next token: `do_sample=False` for greedy decoding, `do_sample=True` for sampling, with `temperature`, `top_k` and `top_p`.

*Write the function `transformers_generate(prompt, length, **settings)`, which generates `length` tokens after `prompt` with `model.generate`, passing it the settings `settings`, and returns the text. Test it with `test_decoding`, with a hundred tokens per prompt: first with top-p sampling restricted to the 30 most likely tokens, with a nucleus `p = 0.95`, then with greedy decoding. Compare their share of distinct 3-grams.*

**Useful documentation**

- [`generate`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationMixin.generate)
- [`GenerationConfig`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationConfig)

In [ ]:
def transformers_generate(prompt: str, length: int, **settings) -> str: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `token_ids = encode(prompt)`, and `torch.ones_like(token_ids)` for the attention mask (`attention_mask`)
- `max_new_tokens=length`: the number of tokens to generate, not counting the prompt
- `max_length=None`: BelGPT-2's generation configuration limits the text to 20 tokens, prompt included; we remove this limit, so that only `max_new_tokens` counts (otherwise, `transformers` warns at each call)
- `pad_token_id=start_token_id, eos_token_id=start_token_id`: the padding and end tokens, which BelGPT-2's configuration gets wrong (see "Inspecting the model")
- `**settings` passes the settings on: `{"do_sample": True, "top_k": 30, "top_p": 0.95}` for top-p sampling, `{"do_sample": False}` for greedy decoding
- `decode(generated)`: `generate` returns a tensor of shape `(1, prompt tokens + length)`, which starts with the prompt

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def transformers_generate(prompt: str, length: int, **settings) -> str:
  token_ids = encode(prompt)
  generated = model.generate(
    token_ids,
    attention_mask=...,
    max_new_tokens=...,
    max_length=None,
    pad_token_id=start_token_id,
    eos_token_id=start_token_id,
    **settings,
  )
  return ...


for settings in [{"do_sample": True, "top_k": 30, "top_p": 0.95}, {"do_sample": False}]:
  generations = test_decoding(transformers_generate, 100, **settings)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"{settings}: {repetition:.2f} distinct 3-grams")
```

### Solution

In [ ]:
def transformers_generate(prompt: str, length: int, **settings) -> str:
  token_ids = encode(prompt)
  generated = model.generate(
    token_ids,
    attention_mask=torch.ones_like(token_ids),
    max_new_tokens=length,
    max_length=None,
    pad_token_id=start_token_id,
    eos_token_id=start_token_id,
    **settings,
  )
  return decode(generated)


for settings in [{"do_sample": True, "top_k": 30, "top_p": 0.95}, {"do_sample": False}]:
  generations = test_decoding(transformers_generate, 100, **settings)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"{settings}: {repetition:.2f} distinct 3-grams")

Over a hundred tokens, the gap widens: most greedy generations end up looping, and their share of distinct 3-grams drops, whereas top-p sampling stays varied. This is why language models are almost always decoded by sampling, with a temperature, a top-k and a top-p tuned for the intended use.